In [49]:
# meeting 2 tasks:
# practice by parsing elexon data from last 24h and combine relevant columns to form the 'matrix'



import pandas as pd

# assign variable to store file path
file_path = 'GenerationByFuelType-2026-06-09T14_00_00.000Z-2026-06-10T14_00_00.000Z.csv'

# reads a csv file into a DataFrame
# a DataFrame is a 2D array and used for larger data sets 
df = pd.read_csv(file_path)



In [50]:
# DataFrame functions:
# prints first 5 rows by default (data appears formatted compared to printing it)
print('First 5 Rows: \n')
print(df.head(100))
print('\n')


print('All Info about DataFrame: \n')
df.info()


First 5 Rows: 

   Dataset           PublishTime             StartTime SettlementDate  \
0   FUELHH  2026-06-10T13:30:00Z  2026-06-10T13:00:00Z     2026-06-10   
1   FUELHH  2026-06-10T13:30:00Z  2026-06-10T13:00:00Z     2026-06-10   
2   FUELHH  2026-06-10T13:30:00Z  2026-06-10T13:00:00Z     2026-06-10   
3   FUELHH  2026-06-10T13:30:00Z  2026-06-10T13:00:00Z     2026-06-10   
4   FUELHH  2026-06-10T13:30:00Z  2026-06-10T13:00:00Z     2026-06-10   
..     ...                   ...                   ...            ...   
95  FUELHH  2026-06-10T11:30:00Z  2026-06-10T11:00:00Z     2026-06-10   
96  FUELHH  2026-06-10T11:30:00Z  2026-06-10T11:00:00Z     2026-06-10   
97  FUELHH  2026-06-10T11:30:00Z  2026-06-10T11:00:00Z     2026-06-10   
98  FUELHH  2026-06-10T11:30:00Z  2026-06-10T11:00:00Z     2026-06-10   
99  FUELHH  2026-06-10T11:30:00Z  2026-06-10T11:00:00Z     2026-06-10   

    SettlementPeriod FuelType  Generation  
0                 29  BIOMASS        2406  
1                 2

In [51]:
# pivot the data (long to wide format) i.e choose index labels (listed along side), column labels (listed across top) and values to fill the table 

# clean column names of empty spaces
df.columns = df.columns.str.strip()

# apply pivot function
df_pivot = df.pivot_table(
    index = ['SettlementDate', 'SettlementPeriod'],
    columns = 'FuelType',
    values = 'Generation',
    aggfunc = sum
)

# turn row labels back into columns
df_pivot = df_pivot.reset_index()



In [52]:
# check the pivoted data

print('First 5 rows: \n')
print(df_pivot.head())
print('\n')

df_pivot.info()

First 5 rows: 

FuelType SettlementDate  SettlementPeriod  BIOMASS  CCGT  COAL  INTELEC  \
0            2026-06-09                30      871  1790     0      996   
1            2026-06-09                31      872  1765     0      996   
2            2026-06-09                32      863  1663     0      996   
3            2026-06-09                33      871  1669     0      996   
4            2026-06-09                34      912  2207     0      996   

FuelType  INTEW  INTFR  INTGRNL  INTIFA2  ...  INTNEM  INTNSL  INTVKL  NPSHYD  \
0             0   1502     -208       -2  ...     902    1398    1248     114   
1             0   1502     -180       -2  ...     898    1398    1148     116   
2             0   1502     -184       -2  ...     898    1398    1154     110   
3             0   1502      -56       -2  ...     996    1398    1412     169   
4             0   1502     -142       -2  ...     996    1398    1426     230   

FuelType  NUCLEAR  OCGT  OIL  OTHER   PS  WIND

In [53]:
# combine relevant columns

# gas column
df_pivot ['Gas'] = df_pivot ['OCGT'] + df_pivot ['CCGT']


# interconnector column
int_columns = df_pivot.filter(like = 'INT') 
#print(int_columns)
df_pivot ['Int'] =  int_columns.sum(axis = 1) #axis=1 means add horizontally (for each row)


# hydroelectric column
df_pivot ['Hydro'] = df_pivot ['NPSHYD'] 

In [54]:
# create the final matrix

# copy columns to the final dataframe
final_matrix_df = df_pivot[
    ['SettlementDate', 
    'SettlementPeriod', 
    'Gas', 
    'NUCLEAR', 
    'Int', 
    'BIOMASS', 
    'Hydro',
    'OTHER']
].copy()

# rename columns
final_matrix_df = final_matrix_df.rename(columns={
    'SettlementDate': 'Date',
    'SettlementPeriod': 'Period',
    'NUCLEAR': 'Nuclear',
    'BIOMASS': 'Biomass',
    'OTHER': 'Other'
})


final_matrix_df = final_matrix_df.sort_values(by=['Date', 'Period']) #sort by date first and then period
print(final_matrix_df.head(48))



FuelType        Date  Period   Gas  Nuclear   Int  Biomass  Hydro  Other
0         2026-06-09      30  1792     2964  6670      871    114    808
1         2026-06-09      31  1767     2966  6684      872    116   1058
2         2026-06-09      32  1664     2871  6588      863    110    745
3         2026-06-09      33  1671     2971  7272      871    169    674
4         2026-06-09      34  2220     2974  7218      912    230    742
5         2026-06-09      35  3238     2971  7114     1254    293    627
6         2026-06-09      36  4164     2972  6734     1505    326    752
7         2026-06-09      37  5883     2970  4560     1903    507   1055
8         2026-06-09      38  6432     2969  4198     2340    636    970
9         2026-06-09      39  7494     2969  3318     2395    647   1301
10        2026-06-09      40  8003     2971  3078     2393    618   1785
11        2026-06-09      41  8116     2972  2592     2393    596   2479
12        2026-06-09      42  8281     2970  2620  